## Methodology
The methodology has been described step by step as follows to classify gender.

### Step 1- Install Classifiers and import all the important libraries.

In [31]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
plt.style.use('fivethirtyeight')
import warnings
warnings.filterwarnings('ignore')

import nltk

import re 

from nltk.stem import PorterStemmer # for stemming

from nltk.stem import WordNetLemmatizer # for lemmatization

from nltk.corpus import stopwords
nltk.download('punkt')

nltk.download('stopwords')
nltk.download('wordnet')
from sklearn.preprocessing import LabelEncoder

from sklearn.feature_extraction.text import CountVectorizer

from sklearn.feature_extraction.text import TfidfVectorizer

from sklearn.model_selection import train_test_split

from sklearn.naive_bayes import GaussianNB

#from xgboost import XGBClassifier

#from lightgbm import LGBMClassifier

from sklearn.metrics import accuracy_score

from sklearn.metrics import confusion_matrix
from sklearn.metrics import accuracy_score

from sklearn.metrics import confusion_matrix
from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\eamon\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\eamon\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\eamon\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


Step 2 - Read data from CSV file

Data set consists of 20050 and 26 columns of tweeter (X) data. 



In [32]:
data = pd.read_csv("gender-classifier.csv",encoding='latin1')

In [5]:
data

,_unit_id,_golden,_unit_state,_trusted_judgments,_last_judgment_at,gender,gender:confidence,profile_yn,profile_yn:confidence,created,...,profileimage,retweet_count,sidebar_color,text,tweet_coord,tweet_count,tweet_created,tweet_id,tweet_location,user_timezone
0,815719226,False,finalized,3,10/26/15 23:24,male,1.0000,yes,1.0,12/5/13 1:48,...,https://pbs.twimg.com/profile_images/414342229...,0,FFFFFF,Robbie E Responds To Critics After Win Against...,NaN,110964,10/26/15 12:40,6.587300e+17,main; @Kan1shk3,Chennai
1,815719227,False,finalized,3,10/26/15 23:30,male,1.0000,yes,1.0,10/1/12 13:51,...,https://pbs.twimg.com/profile_images/539604221...,0,C0DEED,ÛÏIt felt like they were my friends and I was...,NaN,7471,10/26/15 12:40,6.587300e+17,NaN,Eastern Time (US & Canada)
2,815719228,False,finalized,3,10/26/15 23:33,male,0.6625,yes,1.0,11/28/14 11:30,...,https://pbs.twimg.com/profile_images/657330418...,1,C0DEED,i absolutely adore when louis starts the songs...,NaN,5617,10/26/15 12:40,6.587300e+17,clcncl,Belgrade
3,815719229,False,finalized,3,10/26/15 23:10,male,1.0000,yes,1.0,6/11/09 22:39,...,https://pbs.twimg.com/profile_images/259703936...,0,C0DEED,Hi @JordanSpieth - Looking at the url - do you...,NaN,1693,10/26/15 12:40,6.587300e+17,"Palo Alto, CA",Pacific Time (US & Canada)
4,815719230,False,finalized,3,10/27/15 1:15,female,1.0000,yes,1.0,4/16/14 13:23,...,https://pbs.twimg.com/profile_images/564094871...,0,0,Watching Neighbours on Sky+ catching up with t...,NaN,31462,10/26/15 12:40,6.587300e+17,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
20045,815757572,True,golden,259,NaN,female,1.0000,yes,1.0,8/5/15 21:16,...,https://pbs.twimg.com/profile_images/656793310...,0,C0DEED,"@lookupondeath ...Fine, and I'll drink tea too...",NaN,783,10/26/15 13:20,6.587400e+17,Verona ªÁ,NaN
20046,815757681,True,golden,248,NaN,male,1.0000,yes,1.0,8/15/12 21:17,...,https://pbs.twimg.com/profile_images/639815429...,0,0,Greg Hardy you a good player and all but don't...,NaN,13523,10/26/15 12:40,6.587300e+17,"Kansas City, MO",NaN
20047,815757830,True,golden,264,NaN,male,1.0000,yes,1.0,9/3/12 1:17,...,https://pbs.twimg.com/profile_images/655473271...,0,C0DEED,You can miss people and still never want to se...,NaN,26419,10/26/15 13:20,6.587400e+17,Lagos Nigeria,NaN
20048,815757921,True,golden,250,NaN,female,0.8489,yes,1.0,11/6/12 23:46,...,https://pbs.twimg.com/profile_images/657716093...,0,0,@bitemyapp i had noticed your tendency to pee ...,NaN,56073,10/26/15 12:40,6.587300e+17,Texas Hill Country,NaN


## Step 3 - Shape of data

In [6]:
data.shape

(20050, 26)

## Step 4 - Dropping columns

We are dropping some columns we do not need. Wee keep the gender we want to classify, the tweet desciprion and the test itself.

In [7]:
data = pd.concat([data.gender,data.description],axis=1)

### Step 6- Null Values
Check Null values in the dataset.

In [8]:
data.isnull().sum()

gender           97
description    3744
dtype: int64

### Step 7- Drop Null Values
Null values are dropped using dropna() function.

In [9]:
data.dropna(axis=0,inplace=True)

### Step 8- Count the ‘gender’ column.
Count the variables of the ‘gender’ column.

In [10]:
data['gender'].value_counts()

gender
female     5725
male       5469
brand      4328
unknown     702
Name: count, dtype: int64

### Step 9- Save only ‘male’ and ‘female’ variables.
Save only the variables ‘male’ and ‘female’ in the ‘gender’ column as we are concerned about only these two genders and check their count again.

In [11]:
filtered_data = data[data['gender'].isin(['male', 'female'])]

In [12]:
filtered_data['gender'].value_counts()

gender
female    5725
male      5469
Name: count, dtype: int64

In [13]:
filtered_data

,gender,description
0,male,i sing my own rhythm.
1,male,I'm the author of novels filled with family dr...
2,male,louis whining and squealing and all
3,male,"Mobile guy. 49ers, Shazam, Google, Kleiner Pe..."
4,female,Ricky Wilson The Best FRONTMAN/Kaiser Chiefs T...
...,...,...
20045,female,(rp)
20046,male,"Whatever you like, it's not a problem at all. ..."
20047,male,#TeamBarcelona ..You look lost so you should f...
20048,female,Anti-statist; I homeschool my kids. Aspiring t...


### Step 10- Encoding of categorical variables ‘male’ and ‘female’.
Encode the ‘male’ and ‘female’ category as 1 and 0 using the replace() function. Male was encoded as 1 and female

In [14]:
for gen in filtered_data['gender']:
    if gen=='male':
        filtered_data['gender'].replace({'male':'1'},inplace=True)
    elif gen=='female':
        filtered_data['gender'].replace({'female':'0'},inplace=True)
filtered_data['gender'].value_counts()

gender
0    5725
1    5469
Name: count, dtype: int64

### Step11- Cleaning of the description column
keep only the words containing alphanumeric characters and remove punctuations.Defined a function clean() to remove punctuations from the ‘description’ column.

In [15]:
def clean(review):
    descrip = re.sub('[^a-zA-Z]', ' ', review)
    descrip = descrip.lower()
    return descrip

filtered_data['descrip_Cleaned'] = pd.DataFrame(filtered_data['description'].apply(lambda x: clean(x)))
filtered_data.head()

,gender,description,descrip_Cleaned
0,1,i sing my own rhythm.,i sing my own rhythm
1,1,I'm the author of novels filled with family dr...,i m the author of novels filled with family dr...
2,1,louis whining and squealing and all,louis whining and squealing and all
3,1,"Mobile guy. 49ers, Shazam, Google, Kleiner Pe...",mobile guy ers shazam google kleiner pe...
4,0,Ricky Wilson The Best FRONTMAN/Kaiser Chiefs T...,ricky wilson the best frontman kaiser chiefs t...


Cleaning the data which includes punctuation removal,number removal, and different signs like ‘@’,'()’,’#’, and URL with ‘ ‘.

In [16]:
url_regex = r"(https?://)?(www\d?\.)?[a-zA-Z0-9-]+(\.[a-z]{2,})+(/\S*)?"
filtered_data['descrip_Cleaned'] = filtered_data['descrip_Cleaned'].replace(url_regex, "", regex=True)

In [17]:
filtered_data

,gender,description,descrip_Cleaned
0,1,i sing my own rhythm.,i sing my own rhythm
1,1,I'm the author of novels filled with family dr...,i m the author of novels filled with family dr...
2,1,louis whining and squealing and all,louis whining and squealing and all
3,1,"Mobile guy. 49ers, Shazam, Google, Kleiner Pe...",mobile guy ers shazam google kleiner pe...
4,0,Ricky Wilson The Best FRONTMAN/Kaiser Chiefs T...,ricky wilson the best frontman kaiser chiefs t...
...,...,...,...
20045,0,(rp),rp
20046,1,"Whatever you like, it's not a problem at all. ...",whatever you like it s not a problem at all ...
20047,1,#TeamBarcelona ..You look lost so you should f...,teambarcelona you look lost so you should f...
20048,0,Anti-statist; I homeschool my kids. Aspiring t...,anti statist i homeschool my kids aspiring t...


### Step 12- Tokenization of ‘description_Cleaned’ column
Tokenization is the process of breaking text into smaller pieces which we know as tokens. Each word, special character, or number in a sentence can be depicted as a token in NLP. Tokenization is the process of breaking down a piece of code into smaller units called tokens.

Tokenization has been performed using the word_tokenization() function, which splits text into individual words. Tokenized words were stored in a list named descrip_cleaned and after that, a list comprehension was performed using is.alpha() function alphabets were only stored in ‘descrip_new_alpha’ list.

In [18]:
from nltk.tokenize import word_tokenize
filtered_data['descrip_Cleaned'] = [nltk.word_tokenize(tweet) for tweet in filtered_data['descrip_Cleaned']]
descrip_new=[]
for each_row in filtered_data['descrip_Cleaned']:
    descrip_new.append([i for i in each_row if i.isalpha()])
descrip_new_alpha=[]

### Step 13- Stopwords removal from the 'descrip_Cleaned' column.

Stopwords do not add meaning to the sentence, so stop words were removed from the sentence by running a list comprehension and words that do not fall under stopwords were again stored in the list 'descrip_new_alpha'.

In [19]:
stop_words = set(stopwords.words('english'))

for each_row in descrip_new:

    descrip_new_alpha.append([i for i in each_row if i not in stop_words])

### Step14- Lemmatization of the text of the ‘descrip_Cleaned’ column.
Lemmatization is an organized and step-by-step process of obtaining the root form of the word. It makes use of vocabulary and morphological analysis. Lemmatization was done to get to the root of any word and using WordNetlemmatizer() class an object was created through which lemmatization was performed and then using join() function all words were joined into a sentence and the complete cleaned description was stored in a list named ‘descrip_Cleaned’.

In [20]:
description_new_lemma = []

lemma = nltk.WordNetLemmatizer()

for each_row in descrip_new_alpha:

    description_new_lemma.append([lemma.lemmatize(word) for word in each_row])

filtered_data['descrip_Cleaned'] = description_new_lemma

filtered_data['descrip_Cleaned'] = [" ".join(desc) for desc in filtered_data['descrip_Cleaned'].values]

In [21]:
filtered_data

,gender,description,descrip_Cleaned
0,1,i sing my own rhythm.,sing rhythm
1,1,I'm the author of novels filled with family dr...,author novel filled family drama romance
2,1,louis whining and squealing and all,louis whining squealing
3,1,"Mobile guy. 49ers, Shazam, Google, Kleiner Pe...",mobile guy er shazam google kleiner perkins ya...
4,0,Ricky Wilson The Best FRONTMAN/Kaiser Chiefs T...,ricky wilson best frontman kaiser chief best b...
...,...,...,...
20045,0,(rp),rp
20046,1,"Whatever you like, it's not a problem at all. ...",whatever like problem chargernation foreverroy...
20047,1,#TeamBarcelona ..You look lost so you should f...,teambarcelona look lost follow follow heart br...
20048,0,Anti-statist; I homeschool my kids. Aspiring t...,anti statist homeschool kid aspiring thoughtle...


### Step 15- Creating a bag-of-words model (Vectorization)
Vectorization is a methodology in NLP to map words and phrases from vocabulary to a corresponding vector of real numbers which is used to find word predictions, word similarities/semantics. To make documents corpora more relatable for computers they must first be converted into some numerical structure. Few techniques are used to achieve this, is called ‘Bag of Words’.

CountVectorizer is the most straightforward one, which counts the number of times a token shows up in the document and uses this value as its weight. Words are needed to be encoded into integers so that they can be fed to the input of any machine learning model. For this purpose, Scikit-learn’s CountVectorizer() was used to convert a collection of text documents to a vector of term/token counts and maximum features were fixed to 150.

In [ ]:
# %% creating bag of words model
from sklearn.feature_extraction.text import CountVectorizer  # for bag of words 
cv = CountVectorizer(max_features = 150)


cv.get_params() shows the default parameters.

x = cv.fit_transform(filtered_data['descrip_Cleaned']).toarray()
y = filtered_data.iloc[:,0].values  # positive or negative comment

In [ ]:
# from mod text
count_vect = CountVectorizer()
count_vect.get_params() #shows the default parameters.

{'analyzer': 'word',
 'binary': False,
 'decode_error': 'strict',
 'dtype': numpy.int64,
 'encoding': 'utf-8',
 'input': 'content',
 'lowercase': True,
 'max_df': 1.0,
 'max_features': None,
 'min_df': 1,
 'ngram_range': (1, 1),
 'preprocessor': None,
 'stop_words': None,
 'strip_accents': None,
 'token_pattern': '(?u)\\b\\w\\w+\\b',
 'tokenizer': None,
 'vocabulary': None}

In [40]:
# form modujle text
from sklearn.feature_extraction.text import TfidfTransformer
transformer = TfidfTransformer()
# print(TfidfTransformer(transformer))
TfidfTransformer().get_params()
# Default params: {'norm': 'l2', 'smooth_idf': True, 'sublinear_tf': False, 'use_idf': True}
# tdm_tfidf = transformer.fit_transform(tdm)  #  transform the TDM.


{'norm': 'l2', 'smooth_idf': True, 'sublinear_tf': False, 'use_idf': True}

In [23]:
print(x.shape, y.shape)

(11194, 150) (11194,)


### Finally - Train Test Split


In [24]:
# train test split
from sklearn.model_selection import train_test_split

x_train, x_test, y_train, y_test = train_test_split(x,y,test_size = 0.1,random_state = 0)

In [25]:
print(x_train.shape, x_test.shape, y_train.shape, y_test.shape)

(10074, 150) (1120, 150) (10074,) (1120,)


### Model Training

In [26]:
gnbmodel = GaussianNB()
gnbmodel.fit(x_train , y_train)
y_pred = gnbmodel.predict(x_test)
accuracy = accuracy_score(y_test, y_pred)
print("Accuracy: %.2f%%" % (accuracy * 100.0))

Accuracy: 63.04%


In [27]:
confusion_matrix(y_test, y_pred)

array([[487, 102],
       [312, 219]], dtype=int64)

In [28]:
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.61      0.83      0.70       589
           1       0.68      0.41      0.51       531

    accuracy                           0.63      1120
   macro avg       0.65      0.62      0.61      1120
weighted avg       0.64      0.63      0.61      1120



### Step 28- GridsearchCV() was used to tune hyperparameters for three classifiers.
Sometimes we often need to perform turining of training parameters for optimal performance. This increases the accuracy and tunes the model to perform better. There are methods in scikit lern to tune hose parameters autoamtically. It is called grid search. For Naive Bayes classifier the parameters which were passed to GridsearchCV() {‘var_smoothing’: np.logspace(0,-9, num=100)}

In [29]:
param_grid_nb = {
    'var_smoothing': np.logspace(0,-9, num=100)
}
nbModel_grid = GridSearchCV(estimator=gnbmodel, param_grid=param_grid_nb, verbose=1, cv=3, n_jobs=-1)
nbModel_grid.fit(x_train, y_train)

print("Best model parameters:\n", nbModel_grid.best_params_)

#The prediction was done using the best hyperparameter evaluated.

y_pred_hyper = nbModel_grid.predict(x_test)

#Confusion Matrix and Accuracy were determined after Hyperparameter tuning of Naive Bayes Classifier.

print(confusion_matrix(y_test, y_pred_hyper), ": is the confusion matrix")

Fitting 3 folds for each of 100 candidates, totalling 300 fits
Best model parameters:
 {'var_smoothing': 0.0657933224657568}
[[514  75]
 [340 191]] : is the confusion matrix


### Classification report

In [30]:
accuracy_gnb_hyper = accuracy_score(y_test, y_pred_hyper)
print("Accuracy: %.2f%%" % (accuracy_gnb_hyper * 100.0))

Accuracy: 62.95%
